## BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [3]:
import os
import json
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [4]:
MODEL = 'llama3.2'
OLLAMA_BASE_URL = "http://localhost:11434/v1"
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

In [4]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-

## First step: Have LLAMA3.2 figure out which links are relevant

### Use a call to llama3.2 to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [7]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [8]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [9]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://edwarddonner.com/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.

In [10]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = ollama.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [11]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling llama3.2
Found 4 relevant links


{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'company page', 'url': 'https://edwarddonner.com/'},
  {'type': 'careers page', 'url': 'https://edwarddonner.com/'},
  {'type': 'blog posts', 'url': 'https://edwarddonner.com/posts/'}]}

In [10]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling llama3.2
Found 8 relevant links


{'links': [{'type': 'Company page', 'url': 'https://huggingface.co'},
  {'type': 'About us/Changelog', 'url': 'https://changelog.huggingface.co'},
  {'type': 'About us/Brand', 'url': 'https://brand.huggingface.co'},
  {'type': 'About us/Blog', 'url': 'https://blog.huggingface.co'},
  {'type': 'About us/FAQs (not included)',
   'url': 'https://discuss.huggingface.co'},
  {'type': 'Careers/Jobs', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'Enterprise website', 'url': 'https://endpoints.huggingface.co'},
  {'type': 'GitHub repository', 'url': 'https://github.com/huggingface'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to llama3.2

In [14]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [ ]:
print(fetch_page_and_all_relevant_links("https://www.loreal.com/en/"))

Selecting relevant links for https://www.loreal.com/de-de/germany/pages/karriere/ by calling llama3.2
Found 2 relevant links
## Landing Page:

Karriere | L'Oréal Deutschland

Go to Content
Go To Footer
Journalists
Suppliers
Investors & Shareholders
L'Oréal Groupe
stock)
OR.PA
€ 350,00
-0,24%
Group
back
Group
Who We Are
Our purpose
Our purpose
back
Our purpose
Our purpose
Essentiality of Beauty
Strategy & Model
Quality & Safety Standards
Our Performance
Governance & Ethics
Governance & Ethics
back
Governance & Ethics
More About Governance & Ethics
Board of Directors
Executive Committee
Our Ethical Principles
Culture & Heritage
Culture & Heritage
back
Culture & Heritage
More  About Culture & Heritage
Life at L'Oréal
Values & Mindset
L'Oréal History
Newsroom
Newsroom
back
Newsroom
All News & Documentation
Press releases
Latest News
Commitments
back
Commitments
Our Commitments & Responsibilities
For The Planet
For The Planet
back
For The Planet
For The Planet
L'Oréal for the Future
Steward

In [44]:
# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a person portfolio
# and creates a short brochure about the person for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a person portfolio
and creates a short, humorous, entertaining, witty brochure about the person work for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of customers and careers/jobs if you have the information.
"""


In [34]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [37]:
get_brochure_user_prompt("Loreal", "https://www.loreal.com/en/")

Selecting relevant links for https://www.loreal.com/en/ by calling llama3.2
Found 6 relevant links


"\nYou are looking at a company called: Loreal\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nL’Oréal, world leader in beauty : makeup, cosmetics, haircare, perfume\n\nGo to Content\nGo To Footer\nJournalists\nSuppliers\nInvestors & Shareholders\nL'Oréal Groupe\nstock)\nOR.PA\n€ 350.00\n-0.24%\nGroup\nback\nGroup\nWho We Are\nOur purpose\nOur purpose\nback\nOur purpose\nOur purpose\nEssentiality of Beauty\nStrategy & Model\nQuality & Safety Standards\nOur Performance\nGovernance & Ethics\nGovernance & Ethics\nback\nGovernance & Ethics\nMore About Governance & Ethics\nBoard of Directors\nExecutive Committee\nOur Ethical Principles\nCulture & Heritage\nCulture & Heritage\nback\nCulture & Heritage\nMore  About Culture & Heritage\nLife at L'Oréal\nValues & Mindset\nL'Oréal History\nNewsroom\nNewsroom\nback\nNewsroom\nAll News & Documentation\nPress 

In [45]:
def create_brochure(company_name, url):
    response = ollama.chat.completions.create(
        model="llama3.2",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [46]:
create_brochure("Loreal", "https://www.loreal.com/en/")

Selecting relevant links for https://www.loreal.com/en/ by calling llama3.2
Found 8 relevant links


# Discover the Power of Beauty with L'Oréal

At L'Oréal, we believe that beauty is essential to a fulfilling life. That's why we've dedicated our purpose to bringing out the beauty in every person, at every age.

### Our Purpose

Our purpose is simple: to provide high-quality cosmetics and skin care products that make people look and feel their best. We're committed to making Beauty Essential for All People.

### Quality & Safety Standards

At L'Oréal, we don't compromise on quality or safety. Our manufacturing processes are designed to ensure the highest levels of quality control and safety standards are met in every product we launch.

### Brands You Love

From Garnier's natural ingredients to Lancôme's luxurious skincare, our portfolio is home to some of the most trusted beauty brands around the world.

* [Dr.G](link)

* [L'Oréal Paris](link)
* [NYX Professional Makeup](link)
* [Yves Saint Laurent](link)
* [Garnier](link)
* [Maybelline New York](link)
* [Lancôme](link)

### Innovation and Sustainability

We're committed to driving innovation in beauty, while also being mindful of our impact on the planet. Our Accelerator Program is dedicated to finding new solutions for sustainability.

At L'Oréal, we believe that diversity and inclusion are essential to a healthier and more beautiful world.

* [Caring for our Employees](link)
* [Support Communities](link)
* [Promoting Diversity, Equity and Inclusion](link)

### Join the Team

Are you ready to join the L'Oréal team? We're always looking for talented individuals who share our values of innovation, quality, and inclusivity.

[Learn more about our careers page](link)

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [41]:
def stream_brochure(company_name, url):
    stream = ollama.chat.completions.create(
        model="llama3.2",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [43]:
stream_brochure("Loreal", "https://www.loreal.com/en/")

Selecting relevant links for https://www.loreal.com/en/ by calling llama3.2
Found 4 relevant links


**L'Oréal: A Leader in Beauty and Innovation**

We are L'Oréal, a world-renowned leader in the beauty industry, dedicated to enhancing the lives of our customers and contributing to the well-being of our planet.

**Our Purpose**
Our purpose is centered on three fundamental pillars:

*   **Essentiality of Beauty**: We believe that beauty is not just a luxury, but an essential part of life.
*   **Quality & Safety Standards**: We are committed to delivering high-quality products and ensuring their safety for our customers.
*   **Governance & Ethics**: We maintain transparent governance structures and adhere to rigorous ethical standards.

**Our Brands**
We own a diverse portfolio of iconic brands that cater to various beauty segments, including:

*   Consumer Products Division:
    *   L'Oréal Paris
    *   Maybelline New York
    *   Garnier
    *   Lancôme
    *   Yves Saint Laurent
*   Luxe Division:
    *   Kiehl's Since 1851

**Our Commitments**
We are dedicated to making a positive impact on the world through:

*   **For The Planet**: Our initiative to reduce our environmental footprint and promote sustainability.
*   **For The People**: We prioritize diversity, equity, and inclusion in all aspects of our business.

**Join Our Team**
At L'Oréal, we foster a culture that values innovation, teamwork, and excellence. If you share our passion for beauty and making a difference, explore our career opportunities today!

**Investing in Our Future**

Our company's history dates back to 1939 when Marcel Eöhrmann founded Léon Guité with the aim of creating hair dye products using vegetable-based ingredients.

In [48]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("Loreal", "https://www.loreal.com/en/")

Selecting relevant links for https://www.loreal.com/en/ by calling llama3.2
Found 5 relevant links


# Be Beautiful, Inside and Out

Welcome to L'Oréal, the world leader in beauty. For over a century, we've been passionate about making people feel beautiful, inside and out.

Our Purpose
-------------

At L'Oréal, our purpose is to empower individuals to express their unique beauty and to inspire them to live life to the fullest. We believe that beauty is not just skin-deep, but also a state of mind.

## Meet Our Brands

We've built an incredible portfolio of brands in the areas of makeup, cosmetics, haircare, and perfume. Some of our most popular brands include:

* L'Oréal Paris
* Maybelline New York
* Garnier
* Lancôme
* Yves Saint Laurent
* Armani
* Valentino

## Join Our Community

Are you looking for a career that's full of limitless opportunities? Look no further than L'Oréal. We're committed to fostering an inclusive and diverse workplace culture that empowers our employees to grow and thrive.

From researchers and developers to sales professionals and marketers, we offer countless roles across various functions. Check out our careers page to learn more about our current openings and how you can join our talented team.

## Share Our Values

At L'Oréal, we're dedicated to making a positive impact on the world around us. We operate with integrity, transparency, and responsibility. Here are just a few of our core values:

* Respect for human rights
* Sustainability and environmental stewardship
* Diversity, equity, and inclusion
* Quality and safety

## Stand with Us

Make beauty, not a crime. Join L'Oréal as we pledge to reduce animal testing across all our brands.

Investing in You

Join the millions of people who've benefited from our products.

Discover Our Brands

In [50]:
german_brochure_system_prompt = """
Du bist ein professioneller Marketing- und Unternehmensredakteur.

Deine Aufgabe ist es, aus den bereitgestellten Inhalten einer Firmenwebsite
eine neue, gut geschriebene Broschüre auf Deutsch zu erstellen.

Wichtige Regeln:
- Schreibe die Broschüre direkt auf Deutsch.
- Übersetze nicht wörtlich aus dem Englischen.
- Formuliere den Text neu, sodass er natürlich, überzeugend und professionell klingt.
- Nutze nur Informationen, die in den bereitgestellten Website-Inhalten vorkommen.
- Erfinde keine Fakten.
- Schreibe im Stil einer modernen Unternehmensbroschüre.
- Strukturiere die Broschüre mit klaren Überschriften.
- Hebe Produkte, Dienstleistungen, Unternehmenswerte, Kundenfokus und Karrieremöglichkeiten hervor, falls vorhanden.
- Der Text soll ansprechend für potenzielle Kunden, Investoren und Bewerber sein.
- Gib die Antwort in sauberem Markdown zurück.
- Verwende keine Codeblöcke.
"""

# English version (for reference):
# You are a professional marketing and corporate copywriter.
#
# Your task is to create a new, well-written brochure in German
# based on the provided content from a company website.
#
# Important rules:
# - Write the brochure directly in German.
# - Do not translate word-for-word from English.
# - Rewrite the content so it sounds natural, persuasive, and professional.
# - Use only information present in the provided website content.
# - Do not invent any facts.
# - Write in the style of a modern corporate brochure.
# - Structure the brochure with clear headings.
# - Highlight products, services, company values, customer focus, and career opportunities if available.
# - The text should appeal to potential customers, investors, and job candidates.
# - Return the response in clean Markdown.
# - Do not use code blocks.


In [57]:
def translate_brochure_to_german(company_name, url):
    stream = ollama.chat.completions.create(
        model="llama3.2"  ,
        messages=[
            {"role": "system", "content": german_brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
        stream=True
    )
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)    

In [58]:
translate_brochure_to_german("Loreal", "https://www.loreal.com/en/")

Selecting relevant links for https://www.loreal.com/en/ by calling llama3.2
Found 3 relevant links


# Die Geschichte einer Weltreiterin: L'Oréal

Unsere Wurzeln liegen in einem kleinen Parfümerladen in Frankreich, den Léon-Marie-Jacques Perret im Jahr 1909 in Paris eröffnete. Bereits in seinem ersten Jahr entdeckte Perret die Bedeutung von Luxus und Exzeß, was zu einer Entwicklung von hochwertigen Parfümmassen führte.

## Wir entwickeln unsere Strategie

Unsere Strategie ist geprägt vom Begriff "Essentiality of Beauty". Wir glauben, dass Schönheit nicht nur körperliche Erscheinung ist, sondern auch die Erfahrung der eigenen Individualität und die Suche nach Selbstakzeptanz. Deshalb stellen wir uns als Unternehmen zu unserem Ziel mit dir zusammen.

Wir haben eine starke Ausrichtung an Qualität und Sicherheit. Unser Ziel ist es, die höchste Qualität zu erreichen und dabei unsere Kundinnen und Kunden auf den first Geburtstag der neuen Luxusprodukte zu besonderen Erfahrungen einzuläuten.

## Ein führendes Unternehmen in unserem Land

Wir sind das führende Beautyunternehmen der Welt. Unsere Produkte sind in über 140 Ländern erhältlich, um Menschen eine Vielzahl von Angeboten und Möglichkeiten zur Schönheitspflege bieten zu können.

Unser Portfolio reicht von Haarenprodukten bis hin zu Kosmetika und Perfumern: Lancôme, Garnier sind nur einige der Marken in unserem Portfolio. Um den Bedürfnissen unserer Kunden gerecht zu werden, ist auch ein ständiger Innovation gezwungen im Bereich die Gesundheit.

Was uns vereint: 
- Die Leidenschaft für Schönheit
- Der Wunsch nach einer verantwortungsvollen Lösungsfindung 
- Unsere Levensprinzipien für Innovation und Kreativität.
- Eine große Zahl von Möglichkeiten, der unsere Mitarbeiter entdecken können.

Die Menschen bei L'Oréal sind die Schlüssel zu unserem Erfolg. Wir wachsen mit ihnen zusammen und stärken diese Beziehung jeden Tag wieder durch unsere Partnerschaften mit Unternehmen, die uns inspirieren und unterstützen.

Mit neuen Produkten zum Beispiel oder an neuen Orten. Unser Ziel besteht darin, den Menschen mit einem Gefühl von Glück zu servieren.

Unsere Ziele spiegeln unser Engagement für eine gute Zukunft für alle wider. Durch unsere Aktivitäten, Initiativen und Innovationen setzen wir uns ein für eine eine nachhaltige Welt.
Die Welt ist heute und morgen voller Möglichkeiten! 
Wir möchten deine Möglichkeiten nicht übersehen –  - denn bei L'Oréal lernest du nie zu enden.

# Unsere Mission

L’Oréal wendet sich an Kunden aller Altersgruppen, Berufsarten und kulturellen Hintergründe. 

## Produkte in allen Bereich
*   Kosmetika und Haarenprodukte wie: Drogen, Shampoos etc.
*   Hautpflegekurse 
*   Parfüm 

Für eine Schönheit, die auch für immer du existieren möchtest!

# Wer uns findet?

Wer bei L'Oréal arbeiten möchte oder Unternehmen mit uns in Partnerschaften schließen möchte, kann sich hier auf den Jobs- und Unternehmensportalen des Unternehmens nach passenden Stellen informieren.